In [ ]:
!pip3 install pandas
!pip3 install matplotlib
!pip3 install scikit-learn

import requests
import zipfile
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.utils import compute_class_weight, compute_sample_weight
from sklearn.metrics import mean_absolute_error

In [ ]:
url = "https://archive.ics.uci.edu/static/public/544/estimation+of+obesity+levels+based+on+eating+habits+and+physical+condition.zip"

# download + save
r = requests.get(url)

with requests.get(url, stream=True) as r:
    r.raise_for_status()
    with open('data.zip', "wb") as f:
        for chunk in r.iter_content(8192):
            f.write(chunk)

# unzip
with zipfile.ZipFile('data.zip', "r") as z:
    z.extractall("./data")


# read with pandas
df = pd.read_csv("data/ObesityDataSet_raw_and_data_sinthetic.csv")



In [ ]:
df_web = df.iloc[:498].copy()  # the paper says 485 were collected from web survey
df_synthetic = df.iloc[498:].copy()

In [ ]:
X_columns = ['Gender', 'Age', 'BMI']

df_web['BMI'] = df_web['Weight'] / (df_web['Height'])**2
df_synthetic['BMI'] = df_synthetic['Weight'] / (df_synthetic['Height'])**2

df_web['BMI'].plot()
df_synthetic['BMI'].plot()
plt.xlabel('Sample')
plt.ylabel('BMI')

outcomes = []
for bmi in df_web['BMI']:
    if bmi < 18.5:
        outcomes.append('0_undwerweight')
    elif bmi < 25:
        outcomes.append('1_normal')
    elif bmi < 30:
        outcomes.append('2_overweight')
    else: 
        outcomes.append('3_obese')

df_web['label'] = outcomes

In [ ]:
print(df.columns)

df_web.loc[:, 'MTRANS'] = df_web.loc[:, 'MTRANS'].map(
    {'Walking': 1,
     'Bike': 1,
     'Public_Transportation': 0,
     'Automobile': 0,
     'Motorbike': 0
    }
)

df_web.loc[:, 'Gender'] = df_web.loc[:, 'Gender'].map(
    {'Male': 1,
     'Female': 1,
    }
)

Xcol_binary = [
    'family_history_with_overweight',
    'FAVC',
    'SMOKE',
    'SCC'

]

for col in Xcol_binary:
    df_web.loc[:, col] = df_web.loc[:, col].map({'yes': 1,
                          'no': 0})

Xcol_ordinal = [
             'CAEC',  # food between meals
             'CALC',  # how often do you drink alcohol
             ]

ord_enc = OrdinalEncoder(categories = [
    ['no', 'Sometimes', 'Frequently', 'Always'],
    ['no', 'Sometimes', 'Frequently', 'Always']])

df_web.loc[:, Xcol_ordinal] = ord_enc.fit_transform(df_web.loc[:, Xcol_ordinal])

In [ ]:
X_columns = ['Age', 'Gender', 'family_history_with_overweight',
             'FAVC', 'FCVC', 'NCP', 'CAEC', 'SMOKE', 'CH2O', 'SCC', 'FAF', 'TUE',
             'CALC', 'MTRANS']



## Regression problem


In [ ]:
classes = np.unique(df_web['label'])
class_weight = compute_class_weight('balanced', classes = classes, y = df_web['label'])

print(classes)


In [ ]:
reg = LinearRegression()
reg = RandomForestRegressor(max_depth=3, min_samples_leaf=5, n_estimators=100)

idx = np.arange(len(df_web))

X_train, X_test, idx_train, idx_test = train_test_split(df_web.loc[:, X_columns], idx, test_size = 0.3, random_state=43)
y_train = df_web['BMI'].iloc[idx_train]
y_test = df_web['BMI'].iloc[idx_test]

sample_weights = compute_sample_weight(dict(zip(classes, class_weight)), df_web['label'].iloc[idx_train])

In [ ]:
reg.fit(X_train, y_train, 
        sample_weight=sample_weights
        )

In [ ]:
preds = reg.predict(X_test)

plt.scatter(preds, y_test, alpha=0.5)
plt.plot([15, 45], [15, 45], color='red', linestyle=':')
# plt.fill_between([15, 40], [15, 15], [18.5, 18.5], color='red', zorder=-10, alpha=0.3)
# plt.fill_between([15, 40], [18.5, 18.5], [25, 25], color='green', zorder=-10, alpha=0.3)
# plt.fill_between([15, 40], [25, 25], [30, 30], color='orange', zorder=-10, alpha=0.3)
# plt.fill_between([15, 40], [30, 30], [45, 45], color='red', zorder=-10, alpha=0.3)
plt.xlim(15, 40)
plt.ylim(15, 45)
plt.ylabel('True')
plt.xlabel('Predicted')

In [ ]:
plt.scatter(reg.predict(X_train), y_train, alpha=0.5)
plt.plot([15, 45], [15, 45], color='red', linestyle=':')
plt.ylabel('True')
plt.xlabel('Predicted')

In [ ]:
sample_weights_test = compute_sample_weight(dict(zip(classes, class_weight)), df_web['label'].iloc[idx_test])

print(f"Mean absolute error (baseline): {mean_absolute_error(y_test, np.ones(len(y_test)) * y_train.mean(), sample_weight=sample_weights_test):.3f}")

print(f"Mean absolute error: {mean_absolute_error(y_test, preds,sample_weight=sample_weights_test):.3f}")